In [ ]:
from docling.document_converter import DocumentConverter

source = "/media/bharath/DATA_8TB1/Bharath/Problem_Statement_3_e4cc9a3eb9/Problem Statement 1_ For Participants/Aditya Birla(G)_03.pdf"

# 1. Initialize the converter
converter = DocumentConverter()

# 2. Convert the entire document in one go (Preserves internal structure)
result = converter.convert(source)

# 3. Access individual pages from the result object
# Docling's 'result.document' contains the structured data
pages_text = []

# If you want to iterate through the document's pages as Markdown:
# Note: Docling usually exports the whole doc, but we can filter by page index
for page_num, page in result.document.pages.items():
    # This gets the content specifically associated with this page
    page_content = result.document.export_to_markdown(page_no=page_num)
    pages_text.append(page_content)
    print(f"Processed page {page_num}")

# Step 3: Preview
extracted_text = pages_text[0] if pages_text else ""
print(f"\n✅ Extracted {len(pages_text)} pages successfully!")
print("First page preview:", extracted_text[:300] + "...")

In [ ]:
print(extracted_text)


In [ ]:
extracted_text = "\n".join(pages_text)

In [ ]:
print(extracted_text)

In [ ]:
# ! ollama pull qwen2.5:32b


In [ ]:
import os
print([f for f in os.listdir("/media/bharath/DATA_8TB1/Bharath/Problem_Statement_3_e4cc9a3eb9/rulebooks_updated")])
print("\n".join([f for f in os.listdir("/media/bharath/DATA_8TB1/Bharath/Problem_Statement_3_e4cc9a3eb9/rulebooks_updated")]))


In [ ]:
import os
import shutil

def copy_selected_resources(source_folder, target_folder):
    # List of resources determined by knowledge/search to be required for InsurancePlanBundle
    required_files = [
        "StructureDefinition-Organization_updated.json",
        "StructureDefinition-Patient_updated.json",
        "StructureDefinition-Practitioner_updated.json",
        "StructureDefinition-PractitionerRole_updated.json",
        "StructureDefinition-Condition_updated.json",
        "StructureDefinition-Procedure_updated.json",
        "StructureDefinition-DocumentReference_updated.json",
        "StructureDefinition-Binary_updated.json",
        # Including Location as it defines the coverage area for the plan
        "StructureDefinition-Location_updated.json" 
    ]

    if not os.path.exists(target_folder):
        os.makedirs(target_folder)

    copied_count = 0
    for file_name in required_files:
        src = os.path.join(source_folder, file_name)
        dst = os.path.join(target_folder, file_name)

        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f"✅ Copied: {file_name}")
            copied_count += 1
        else:
            print(f"⚠️ Skipping: {file_name} (Not found in source)")

    print(f"\nDone! Successfully moved {copied_count} key resources to the NHCX folder.")

# Paths provided
source_dir = "/media/bharath/DATA_8TB1/Bharath/Problem_Statement_2_630a8c8cb6/rulebooks_updated"
target_dir = "/media/bharath/DATA_8TB1/Bharath/Problem_Statement_3_e4cc9a3eb9/rulebooks_updated"

copy_selected_resources(source_dir, target_dir)

In [ ]:
nhcx_extraction_dictionary = {
    "NHCXArtifact": {
        "InsurancePlanBundle": "This profile is based on a Bundle of type collection, providing a description of a health insurance package that consists of a comprehensive list of covered benefits (referred to as the product), associated costs (known as the plan), and supplementary details regarding the offering, such as ownership and administration."
    },
    "OtherResources": {
        "InsurancePlan": "Represents the health insurance product/plan provided by an organization. It describes the contractual arrangement, covered benefits (product), and cost-sharing structures (plan) offered to consumers.",
        "Claim": "A provider-issued list of professional services and products provided, or to be provided, to a patient. It is sent to an insurer for reimbursement, preauthorization, or predetermination.",
        "ClaimResponse": "This resource provides the adjudication results from a payer (insurer) in response to a Claim resource, detailing payments, rejections, and amounts for each line item.",
        "Coverage": "This profile sets the minimum expectations for the Coverage resource to record and search for insurance plan details for a patient, linking the beneficiary to a specific insurance policy.",
        "CoverageEligibilityRequest": "Used by healthcare providers to check with a payer whether a patient has insurance coverage for specific services and to discover the terms of that coverage.",
        "CoverageEligibilityResponse": "The response from a payer providing eligibility and plan details (like remaining deductibles or authorization requirements) following a CoverageEligibilityRequest.",
        "Task": "In the NHCX context, this resource is used to convey information related to payments, status checks during claim adjudication, and facilitating the request or transmission of supporting documentation.",
        "Communication": "A record of an exchange of information between a sender and a receiver (e.g., provider and payer), used to document any communication that occurred during the claims process.",
        "CommunicationRequest": "A record of a request for a communication to take place, such as a payer requesting additional documents from a provider to process a claim.",
        "PaymentNotice": "A notification that a payment has been made or a payment status has changed, confirming to the payee that funds have been transferred.",
        "PaymentReconciliation": "Used to reconcile a bulk payment (e.g., a single bank transfer) against multiple individual claims, providing a detailed breakdown of the total amount settled.",
        "Organization": "Sets minimum expectations for the Organization resource to record, search, and fetch information about healthcare organizations, insurers, or TPAs.",
        "Patient": "Sets minimum expectations for the Patient resource to record, search, and fetch basic demographics and administrative information about an individual beneficiary.",
        "Practitioner": "Sets minimum expectations for the Practitioner resource to record, search, and fetch demographics and administrative info about a healthcare professional.",
        "PractitionerRole": "Describes the specific roles, specialties, and locations of a practitioner within an organization (e.g., a surgeon at a specific hospital).",
        "Condition": "Used to record a list of conditions, problems, or diagnoses associated with a patient, often used in claims to justify medical necessity.",
        "Procedure": "Records details of clinical actions or procedures performed on a patient, which are mapped to line items in a claim for reimbursement.",
        "DocumentReference": "Provides a reference to a document (like a clinical note or lab report) to support the claim, acting as a pointer to the actual data artifact.",
        "Binary": "Allows for the storage and retrieval of raw digital content (like a scanned PDF of a diagnostic report or an insurance brochure) in its native format.",
    }
}

In [ ]:

from langchain_ollama import ChatOllama


llm = ChatOllama(
    model="qwen2.5:32b", 
    temperature=0,
    # num_predict is the "max tokens" for the output. 
    # Setting this to 8192 prevents the "EOF" error.
    num_predict=8192, 
    # num_ctx is the input memory. 
    # 32k is usually plenty for clinical text.
    num_ctx=32768
)


In [ ]:
import math
from langchain_core.messages import HumanMessage

import math
from langchain_core.messages import HumanMessage

def distill_insurance_text(full_text, llm):
    # 1. Divide text with OVERLAP to prevent data loss at boundaries
    num_chunks = 8 # Increased chunks slightly for better focus
    overlap_size = 2000 # ~500 words overlap
    
    total_len = len(full_text)
    chunk_size = math.ceil(total_len / num_chunks)
    
    chunks = []
    for i in range(num_chunks):
        start = max(0, i * chunk_size - overlap_size)
        end = min(total_len, (i + 1) * chunk_size)
        chunks.append(full_text[start:end])
    
    distilled_outputs = []
    
    # 2. The "Lossless" Distillation Prompt
    # This prompt focuses on capturing facts rather than filling a form
    distill_prompt_template = """
    ACT AS an Insurance Policy Underwriter. Your goal is to simplify this policy text into a high-density "Fact Sheet" for a FHIR Architect.

    TASK:
    Scan the text below and extract EVERY technical detail related to insurance policy parameters. 

    STRICT EXTRACTION RULES:
    1. CAPTURE ALL NUMERICS: Every INR value, Percentage (%), Day limit, or Age limit must be preserved.
    2. PRESERVE TABLES: If you find a benefit table or a list of limits (e.g., Room Rent, ICU), recreate it as a Markdown Table.
    3. NO NARRATIVE: Do not use filler words like "This section discusses...". Just state the facts.
    4. NO "NOT SPECIFIED": If a specific category (like TPA) isn't there, simply MOVE ON. Do not write "Not specified".
    5. TERMINOLOGY: Keep specific medical/insurance terms (e.g., "Pre-existing Disease", "OPD", "Co-payment", "Waiting Period").
    6. INSURANCE PLAN DETAILS: Don't miss on extracting details about the insurance plan, covered benefits, costs, and any specific limits or conditions mentioned.

    TEXT CONTENT:
    {chunk_text}

    OUTPUT:
    Provide a condensed version of the relevant data found above. If the text is purely legal preamble with no specific limits or benefits, return: [NO_INSURANCE_DATA]
    """

    print(f"🚀 Distilling {total_len} characters with overlap...")

    for i, chunk in enumerate(chunks):
        print(f"📝 Processing Section {i+1}/{num_chunks}...")
        
        try:
            response = llm.invoke([HumanMessage(content=distill_prompt_template.format(chunk_text=chunk))])
            content = response.content.strip()
            
            if "[NO_INSURANCE_DATA]" not in content:
                distilled_outputs.append(f"### SECTION {i+1} SUMMARY ###\n{content}")
        
        except Exception as e:
            print(f"❌ Error in Section {i+1}: {e}")

    # 3. Join the distilled facts
    final_distilled_text = "\n\n".join(distilled_outputs)
    
    print(f"✅ Distillation complete.")
    print(f"Original: {total_len} chars | Distilled: {len(final_distilled_text)} chars")
    return final_distilled_text

# Execute the process
distilled_text = distill_insurance_text(extracted_text, llm)

In [ ]:
print(distilled_text)

In [ ]:
import json
import re

# Deterministic mapping of mandatory resources based on the selected artifact
def get_must_resources(artifact):
    if artifact == "InsurancePlanBundle":
        return [
            "InsurancePlanBundle", "InsurancePlan", "Organization", "Condition", "DocumentReference"
        ]

    return []

# The Optimized Prompt
prompt = f"""
ACT AS an expert NHCX FHIR Data Architect.

**TASK:**
Identify and select relevant FHIR resources from the `OtherResources` section of the provided `nhcx_extraction_dictionary` based **ONLY** on the clinical and administrative information present in the `[Extracted Text]`.

**RULES:**
1. **Source Strictly from Text**: Only select a resource if the `[Extracted Text]` contains specific data points (e.g., policy numbers, diagnosis names, procedure names, insurer names, patient names) that belong in that resource profile.
2. **Exclude Mandatory Base**: DO NOT select resources that are already part of the Mandatory Base for the primary artifact (e.g., if the primary is `InsurancePlanBundle`, do not list `InsurancePlan` or `Organization` as "Other" as they are already core components).
3. **Accuracy**: If the text mentions a "Diagnosis," select `Condition`. If it mentions "Surgery," select `Procedure`. If it mentions "Co-pay/Policy details," select `Coverage`. If no relevant information is found for a category, return an empty list.

**INPUT:**
[Extracted Text]: 
{distilled_text}

[Dictionary]:
{json.dumps(nhcx_extraction_dictionary, indent=2)}

**OUTPUT FORMAT:**
Return ONLY a valid JSON object. No pre-amble, markdown blocks, or explanation.
{{
    "selected_other_resources": ["Key1", "Key2", ...]
}}
"""

# Invoke the LLM
response = llm.invoke(prompt)
raw_output = response.content.strip()

# Parsing Logic
try:
    # Clean up potential markdown formatting
    clean_json = re.sub(r'^```json\s*|```$', '', raw_output, flags=re.MULTILINE).strip()
    data = json.loads(clean_json)
    
    # Final Variables
    clinical_artifact = "InsurancePlanBundle"  # This is fixed based on the problem statement
    must_resources = get_must_resources(clinical_artifact)
    selected_other_resources = data.get("selected_other_resources", [])
    selected_other_resources = [res for res in selected_other_resources 
                          if res not in must_resources]
    
    # Logging the results
    print(f"--- Extraction Complete ---")
    print(f"Artifact: {clinical_artifact}")
    print(f"Must Resources (Fixed): {must_resources}")
    print(f"Other Selected Resources: {selected_other_resources}")

except Exception as e:
    print(f"Error parsing LLM output: {e}")
    print(f"Raw response was: {raw_output}")

In [ ]:
# RESOURCE_DEPENDENCIES = {
#     "Patient": [],
#     "Organization": [],
#     "Practitioner": ["Patient"],
#     "PractitionerRole": ["Patient", "Practitioner", "Organization"],
#     "Procedure": ["Patient", "Practitioner", "PractitionerRole"],
#     "Condition": ["Patient", "PractitionerRole"],
#     "DocumentReference": ["Patient"],
#     "Claim": ["Patient", "Practitioner", "Organization", "Coverage"],
#     "ClaimResponse": ["Patient", "Claim"],
#     "Coverage": ["Patient", "Organization"],
#     "CoverageEligibilityRequest": ["Patient", "Coverage"],
#     "CoverageEligibilityResponse": ["Patient", "CoverageEligibilityRequest"],
#     "InsurancePlan": ["Organization"],
#     "InsurancePlanBundle": ["Patient", "InsurancePlan", "Coverage"],
#     "Communication": ["Patient", "Practitioner"],
#     "CommunicationRequest": ["Patient", "Practitioner"],
#     "Task": ["Patient", "Practitioner"],
#     "PaymentNotice": ["Claim", "PaymentReconciliation"],
#     "PaymentReconciliation": ["Claim", "ClaimResponse"],
#     "Binary": ["DocumentReference"]
# }

# Use this focused dictionary for Insurance Plan Extraction
RESOURCE_DEPENDENCIES = {
    "Organization": [],
    "Binary": [],
    "DocumentReference": ["Binary"],
    "InsurancePlan": ["Organization"],
    "HealthcareService": ["InsurancePlan"],
    "InsurancePlanBundle": ["InsurancePlan", "Organization"] 
    # Notice: No Patient, No Coverage, No Claim needed here!
}


In [ ]:
import json
import uuid
import operator
from typing import TypedDict, List, Dict, Annotated, Any
from langgraph.graph import StateGraph, END
# from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import os
from datetime import datetime, timezone


# ---------------- STATE ----------------
class AgentState(TypedDict):
    text: str
    id_registry: Dict[str, Any]
    final_resources: Annotated[List[dict], operator.add]
    rulebook_paths: Dict[str, str]

# ---------------- LLM ----------------
# llm = ChatOllama(model="qwen2.5:latest", temperature=0)
# llm = ChatOllama(model="deepseek-coder-v2", temperature=0)



# ---------------- JSON EXTRACTION ----------------
def extract_json(text: str):
    if not text or not text.strip():
        return None
    
    # Remove markdown code blocks
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    text = text.strip()
    
    decoder = json.JSONDecoder()
    idx = 0
    
    while idx < len(text):
        try:
            obj, end = decoder.raw_decode(text[idx:])
            if isinstance(obj, str):
                try:
                    obj = json.loads(obj)
                except:
                    pass
            return obj
        except json.JSONDecodeError:
            idx += 1
    return None

# ---------------- NORMALIZE FUNCTIONS ----------------
def ensure_id(resource):
    if not isinstance(resource, dict):
        return resource
    if "id" not in resource or not resource["id"]:
        resource["id"] = str(uuid.uuid4())
    return resource

def normalize_resource_output(res, resource_type):
    """Convert any input to single dict or list of dicts."""
    if isinstance(res, str):
        parsed = extract_json(res)
        if parsed:
            res = parsed
    
    if isinstance(res, dict):
        return [res]
    elif isinstance(res, list):
        return res
    else:
        # Create minimal resource
        return [{
            "resourceType": resource_type,
            "id": str(uuid.uuid4()),
            "meta": {"profile": [f"https://nrces.in/ndhm/fhir/r4/StructureDefinition/{resource_type}"]}
        }]

def get_single_resource(resources_list, resource_type):
    """Get first valid resource from list."""
    for res in resources_list:
        if isinstance(res, dict) and res.get("resourceType") == resource_type:
            return res
    # Return first item or create new
    if resources_list:
        res = resources_list
        if isinstance(res, dict):
            res["resourceType"] = resource_type
            return res
    return {
        "resourceType": resource_type,
        "id": str(uuid.uuid4()),
        "meta": {"profile": [f"https://nrces.in/ndhm/fhir/r4/StructureDefinition/{resource_type}"]}
    }


In [ ]:
# ---------------- CORE AGENT FUNCTION ----------------
def run_extraction_agent(state: AgentState, resource_type: str):
    rulebook_path = state['rulebook_paths'].get(resource_type)
     # Load rulebook content
    rulebook_content = ""
    if rulebook_path and os.path.exists(rulebook_path):
        with open(rulebook_path, 'r', encoding='utf-8') as f:
            rulebook_content = f.read()
    
    prompt = f'''
    ACT AS an expert NHCX FHIR Data Architect. 

EXTRACT ONLY a valid HL7 FHIR R4 {resource_type} resource (or a Bundle containing multiple resources) from the provided technical insurance text.

RULEBOOK (STRUCTURE GUIDANCE):
{rulebook_content}

INSURANCE POLICY TEXT (DISTILLED):
{state["text"]}

STRICT REQUIREMENTS (NON-NEGOTIABLE):
• Output MUST be valid JSON only.
• Output MUST start with "{{" or "[".
• DO NOT output markdown code fences (e.g., no ```json), no preamble, no comments, and no explanations.
• DO NOT hallucinate or infer missing data. If a field (like TPA name or specific Co-pay) is not in the text, OMIT IT.
• Extract ONLY information explicitly present in the provided text.
• Omit any field whose value is not clearly present.

NHCX + ABDM CONSTRAINTS:
• Conform to NHCX (National Health Claims Exchange) and ABDM profiling expectations.
• Resource Type: If extracting multiple linked resources, wrap them in a Bundle of type "collection".
• Identifiers: Every resource MUST contain an "id" as a UUID string (RFC-4122 format).
• Use the Product UIN (e.g., ADIHLGP22023V032122) as the business 'identifier' for the InsurancePlan resource.
• DO NOT include empty objects, empty arrays, or null values.

TERMINOLOGY & CODING RULES:
• Use IRDAI Standard Exclusion Codes (e.g., Excl03, Excl04) for exclusions.
• Use SNOMED CT for clinical conditions (e.g., Cancer, Myocardial Infarction) if coding is required.
• System URLs:
  - IRDAI Exclusions -> [https://irdai.gov.in/exclusions](https://irdai.gov.in/exclusions)
  - SNOMED CT -> [http://snomed.info/sct](http://snomed.info/sct)
• If no explicit code exists in the text, use only the "text" attribute within the CodeableConcept.
• NEVER fabricate codes.

REFERENCE & LINKING RULES:
• Use URN UUID references for internal Bundle linking: "reference": "urn:uuid:<uuid-here>".
• The InsurancePlan resource MUST reference the 'Organization' (Payer) via the .ownedBy element.
• The InsurancePlan resource SHOULD reference 'Location' resources for network/excluded hospitals if data is present.
• Only create references explicitly justified by the text.

DATA ACCURACY RULES:
• Preserve numeric precision exactly (e.g., 7.5 dioptres, 150% pay-out).
• Preserve all currency values (INR) and time-based limits (Waiting Periods) exactly.
• Ensure "Exclusions" are mapped correctly to either the general plan level or specific benefit level.

OUTPUT FORMAT:
Return ONLY the JSON resource(s) for {resource_type}.
'''

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        raw_output = response.content.strip()
        print(f"\n🔍 Raw output for {resource_type}:\n{raw_output[:500]}...")
        
        parsed = extract_json(raw_output)
        if parsed:
            return parsed
        
        print(f"⚠️ Could not parse JSON for {resource_type}")
        
    except Exception as e:
        print(f"❌ Error for {resource_type}: {e}")
    
    # Fallback minimal resource
    return [{
        "resourceType": resource_type,
        "id": str(uuid.uuid4()),
        "meta": {"profile": [f"https://nrces.in/ndhm/fhir/r4/StructureDefinition/{resource_type}"]}
    }]


In [ ]:

_node_cache = {}

def create_insurance_node(resource_type: str):
    """
    Factory function for NHCX Insurance resources.
    Handles Bundle creation for InsurancePlanBundle and individual financial resources.
    """
    if resource_type in _node_cache:
        return _node_cache[resource_type]
    
    def node(state: AgentState):
        # ✅ SPECIAL CASE: InsurancePlanBundle is a Bundle resource
        if resource_type == "InsurancePlanBundle":
            actual_resource_type = "Bundle"
            is_insurance_bundle = True
        else:
            actual_resource_type = resource_type
            is_insurance_bundle = False
        
        # Run the extraction agent (using the prompt we defined previously)
        resources = run_extraction_agent(state, actual_resource_type)
        resources = normalize_resource_output(resources, actual_resource_type)
        
        if isinstance(resources, list):
            safe_resources = []
            # Bundle should typically be singular, but constituent resources can be multiple
            max_items = 1 if is_insurance_bundle else 15 
            for res in resources[:max_items]:
                if isinstance(res, dict):
                    if res.get("resourceType") != actual_resource_type:
                        res["resourceType"] = actual_resource_type
                    
                    # ✅ NHCX Profile forcing
                    if is_insurance_bundle:
                        res.setdefault('meta', {})['profile'] = [
                            "https://nrces.in/ndhm/fhir/r4/StructureDefinition/InsurancePlanBundle"
                        ]
                        res['type'] = 'collection' # Mandatory for NHCX InsurancePlanBundle
                    
                    res = ensure_id(res)
                    
                    # Add Payer/Organization reference logic
                    # If we have a payer_id in registry, link the InsurancePlan to it
                    payer_id = state['id_registry'].get('organization_id')
                    if payer_id and resource_type == "InsurancePlan":
                        res['ownedBy'] = {'reference': f'urn:uuid:{payer_id}'}
                    
                    safe_resources.append(res)
            result = safe_resources
        else:
            result = get_single_resource([resources], actual_resource_type)
            result = ensure_id(result)
            
            # Profile forcing for single object return
            if is_insurance_bundle:
                result.setdefault('meta', {})['profile'] = [
                    "https://nrces.in/ndhm/fhir/r4/StructureDefinition/InsurancePlanBundle"
                ]
                result['type'] = 'collection'
            
            # Linkage logic for single resource
            payer_id = state['id_registry'].get('organization_id')
            if payer_id and resource_type == "InsurancePlan":
                result['ownedBy'] = {'reference': f'urn:uuid:{payer_id}'}
        
        # ✅ ID REGISTRATION LOGIC
        # Store refs differently for the bundle vs individual resources
        if isinstance(result, list):
            state['id_registry'][f'{resource_type.lower()}_refs'] = [
                {'reference': f'urn:uuid:{r["id"]}'} for r in result
            ]
            # If it's the primary organization, save its ID specifically for linking
            if resource_type == "Organization" and len(result) > 0:
                state['id_registry']['organization_id'] = result[0]['id']
        else:
            state['id_registry'][f'{resource_type.lower()}_id'] = result['id']
            if resource_type == "Organization":
                state['id_registry']['organization_id'] = result['id']
        
        count = len(result) if isinstance(result, list) else 1
        resource_display = "Bundle" if is_insurance_bundle else resource_type
        print(f"✅ {resource_type}: {count} {resource_display}")
        
        return {"final_resources": [result] if isinstance(result, list) else [result]}
    
    node.__name__ = f"{resource_type.lower()}_node"
    _node_cache[resource_type] = node
    return node

# ✅ CLEAR CACHE BETWEEN WORKFLOWS (if needed)
def clear_node_cache():
    global _node_cache
    _node_cache = {}


In [ ]:
def insurance_assembly_node(state):
    """
    Assembles the final NHCX InsurancePlanBundle.
    Structure: InsurancePlan FIRST, supporting resources MIDDLE, DocumentReference/Binary LAST.
    """
    import uuid
    from datetime import datetime, timezone

    bundle = {
        "resourceType": "Bundle",
        "id": str(uuid.uuid4()),
        "meta": {
            "profile": ["https://nrces.in/ndhm/fhir/r4/StructureDefinition/InsurancePlanBundle"]
        },
        "type": "collection", # NHCX InsurancePlanBundle must be a collection
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "entry": []
    }
    
    # ✅ STEP 1: Find and Add the InsurancePlan FIRST
    # This serves as the 'anchor' of the bundle
    plan_found = False
    seen_ids = set()

    for resources_list in state["final_resources"]:
        if isinstance(resources_list, list):
            for res in resources_list:
                if isinstance(res, dict) and res.get('resourceType') == 'InsurancePlan':
                    bundle["entry"].insert(0, {
                        "fullUrl": f"urn:uuid:{res['id']}",
                        "resource": res
                    })
                    seen_ids.add(res['id'])
                    plan_found = True
                    print(f"✅ InsurancePlan ({res['id']}) added FIRST")
                    break
        if plan_found:
            break

    # ✅ STEP 2: Categorize remaining resources
    # We want to keep DocumentReference and Binary for the end
    supporting_entries = []
    attachment_entries = []
    
    for resources_list in state["final_resources"]:
        if not isinstance(resources_list, list):
            resources_list = [resources_list]
            
        for r in resources_list:
            if not isinstance(r, dict) or r.get('id') in seen_ids:
                continue
            
            resource_type = r.get('resourceType')
            entry = {
                "fullUrl": f"urn:uuid:{r['id']}",
                "resource": r
            }
            
            # Group Binary and DocumentReference to be added last
            if resource_type in ['DocumentReference', 'Binary']:
                attachment_entries.append(entry)
            else:
                supporting_entries.append(entry)
            
            seen_ids.add(r['id'])

    # ✅ STEP 3: Assemble the entries in order
    # 1. (Already added InsurancePlan at index 0)
    # 2. Add Supporting Resources (Organization, Location, HealthcareService)
    bundle["entry"].extend(supporting_entries)
    
    # 3. Add Attachments (The original PDF data) LAST
    bundle["entry"].extend(attachment_entries)

    print(f"✅ NHCX Bundle Assembled: {len(bundle['entry'])} total resources.")
    print(f"📊 Breakdown: 1 InsurancePlan, {len(supporting_entries)} Supporting, {len(attachment_entries)} Attachments.")
    
    return {"final_resources": [bundle]}

In [ ]:
def build_insurance_workflow(clinical_artifact: str, selected_other_resources: List[str], rulebook_paths: Dict[str, str]):
    # 1. Get mandatory resources for InsurancePlanBundle (Organization, InsurancePlan, etc.)
    must_resources = get_must_resources(clinical_artifact)
    
    # 2. Filter out duplicates
    selected_other_resources = [res for res in selected_other_resources if res not in must_resources]
    all_resources = list(set(must_resources + selected_other_resources))
    
    # Ensure the main artifact (InsurancePlanBundle) is included if not already
    if clinical_artifact not in all_resources:
        all_resources.append(clinical_artifact)
    
    print(f"📋 NHCX Workflow for {clinical_artifact}: {all_resources}")
    
    workflow = StateGraph(AgentState)
    
    # ✅ CREATE NODES
    created_nodes = set()
    for resource in all_resources:
        node_name = resource.lower()
        if node_name not in created_nodes:
            # Using the insurance factory function we created earlier
            node_func = create_insurance_node(resource) 
            workflow.add_node(node_name, node_func)
            created_nodes.add(node_name)
            print(f"✅ Added node: {node_name}")
    
    # 3. Topological sort (Uses your RESOURCE_DEPENDENCIES)
    def topological_sort(resources):
        visited = set()
        order = []
        def visit(resource):
            if resource in visited: return
            visited.add(resource)
            for dep in RESOURCE_DEPENDENCIES.get(resource, []):
                if dep in resources:
                    visit(dep)
            order.append(resource)
        for resource in resources:
            visit(resource)
        return order
    
    resource_order = topological_sort(all_resources)
    print(f"📊 Execution order: {[r.lower() for r in resource_order]}")
    
    # 4. Create Edges
    for i in range(len(resource_order) - 1):
        current = resource_order[i].lower()
        next_node = resource_order[i + 1].lower()
        workflow.add_edge(current, next_node)
        print(f"➡️  Edge: {current} → {next_node}")
    
    # 5. Assembly Node (Using the insurance_assembly_node created earlier)
    workflow.add_node("assembly", insurance_assembly_node)
    last_node = resource_order[-1].lower()
    workflow.add_edge(last_node, "assembly")
    workflow.add_edge("assembly", END)
    
    # ✅ FIX: Dynamic Entry Point
    # In NHCX, 'organization' (the Payer) is usually the best starting point
    if "organization" in created_nodes:
        workflow.set_entry_point("organization")
    else:
        # Fallback to the first resource in the sorted order
        workflow.set_entry_point(resource_order[0].lower())
    
    return workflow.compile(), all_resources

In [ ]:
def run_nhcx_insurance_pipeline(distilled_text: str, clinical_artifact: str, selected_other_resources: List[str]):
    """
    NHCX Pipeline: 
    1. Distills 32-page text into a high-density insurance fact sheet.
    2. Runs the dynamic graph on the distilled text.
    """
    
    
    # --- STEP 2: RULEBOOK MAPPING ---
    rulebook_paths = {
        "Organization": "/media/bharath/DATA_8TB1/Bharath/Problem_Statement_3_e4cc9a3eb9/rulebooks_updated/StructureDefinition-Organization_updated.json",
        "InsurancePlan": "/media/bharath/DATA_8TB1/Bharath/Problem_Statement_3_e4cc9a3eb9/rulebooks_updated/StructureDefinition-InsurancePlan_updated.json",
        "InsurancePlanBundle": "/media/bharath/DATA_8TB1/Bharath/Problem_Statement_3_e4cc9a3eb9/rulebooks_updated/StructureDefinition-InsurancePlanBundle_updated.json",
        **{
            res: f"/media/bharath/DATA_8TB1/Bharath/Problem_Statement_3_e4cc9a3eb9/rulebooks_updated/StructureDefinition-{res}_updated.json"
            for res in selected_other_resources
        }
    }
    
    # --- STEP 3: INITIAL STATE ---
    # We use the DISTILLED text here, not the full_extracted_text
    initial_state = {
        "text": distilled_text, 
        "clinical_artifact": clinical_artifact,
        "id_registry": {},
        "final_resources": [],
        "rulebook_paths": rulebook_paths
    }
    
    # --- STEP 4: WORKFLOW COMPILATION ---
    app, used_resources = build_insurance_workflow(clinical_artifact, selected_other_resources, rulebook_paths)
    
    print(f"🚀 Starting NHCX Bundle Generation for {clinical_artifact}...")
    final_output = app.invoke(initial_state)
    
    # The last resource in final_resources is the assembled Bundle
    bundle = final_output['final_resources'][-1]
    
    # --- STEP 5: SAVE OUTPUT ---
    filename = f"nhcx_{clinical_artifact.lower()}_bundle.json"
    with open(filename, "w") as f:
        json.dump(bundle, f, indent=2)
    
    print(f"\n🎉 SUCCESS! NHCX Bundle saved to {filename}")
    print(f"📊 Resources processed: {used_resources}")
    print(f"📦 Bundle entries: {len(bundle.get('entry', []))}")
    
    return bundle

# # Usage example (from your LLM classification)
# clinical_artifact = "DiagnosticReportRecord"  # From your LLM
# must_resources = get_must_resources(clinical_artifact)
# selected_other_resources = ["ObservationVitalSigns"]  # From your LLM

bundle = run_nhcx_insurance_pipeline(distilled_text, clinical_artifact, selected_other_resources)


In [ ]:
from docling.document_converter import DocumentConverter
import re
from collections import defaultdict

def extract_metadata(page_text):
    """
    Extract key metadata used for grouping.
    We use Age/Sex + Collection Date (DATE ONLY) as primary fingerprint.
    """

    # Extract Age/Sex
    age_sex_match = re.search(r'Age/Sex\s*:\s*(.*)', page_text)
    age_sex = age_sex_match.group(1).strip() if age_sex_match else "UNKNOWN"

    # Extract ONLY date part (ignore time)
    collection_match = re.search(
        r'Collection Date\s*:\s*([0-9]{2}-[A-Za-z]{3}-[0-9]{4})',
        page_text
    )
    collection_date = collection_match.group(1) if collection_match else "UNKNOWN"

    # Extract Lab No (optional for debugging)
    lab_no_match = re.search(r'Lab No\.\s*:\s*(.*)', page_text)
    lab_no = lab_no_match.group(1).strip() if lab_no_match else "UNKNOWN"

    print(f"Extracted Metadata - Age/Sex: {age_sex}, Collection Date: {collection_date}, Lab No: {lab_no}")

    return age_sex, collection_date

def group_pages_by_patient(pages_text):
    """
    Group pages belonging to same patient using strong fingerprint.
    """

    grouped = defaultdict(list)

    for page_number, page_text in enumerate(pages_text, start=1):
        age_sex, collection_date = extract_metadata(page_text)

        fingerprint = f"{age_sex}_{collection_date}"

        grouped[fingerprint].append((page_number, page_text))

    final_patient_texts = []

    print("\n📌 GROUPING SUMMARY")
    print("=" * 50)

    for patient_index, (key, page_data) in enumerate(grouped.items(), start=1):

        page_numbers = [str(page_num) for page_num, _ in page_data]
        merged_text = "\n\n".join([text for _, text in page_data])

        final_patient_texts.append(merged_text)

        age_sex, collection_date = key.split("_", 1)

        if len(page_numbers) > 1:
            print(f"🟢 Patient {patient_index} ({age_sex}, {collection_date})")
            print(f"   → Merged Pages: {', '.join(page_numbers)}")
        else:
            print(f"🔵 Patient {patient_index} ({age_sex}, {collection_date})")
            print(f"   → Single Page: {page_numbers[0]}")

        print("-" * 50)

    # print(f"\n🎯 Total Unique Patients Identified: {len(final_patient_texts)}\n")

    return final_patient_texts

def process_pdf_and_group_patients(pdf_path):
    """
    MAIN FUNCTION

    Input:
        pdf_path (str)

    Output:
        list of unique patient text blocks
    """

    # Step 1: Convert using Docling
    converter = DocumentConverter()
    result = converter.convert(pdf_path)

    # Step 2: Extract page-wise text
    pages_text = []

    for i, page_num in enumerate(result.document.pages.keys(), start=1):
        page_content = result.document.export_to_markdown(page_no=page_num)
        pages_text.append(page_content)
        print(f"Processed page {i}")

    print(f"\n✅ Extracted {len(pages_text)} pages successfully!")

    # Step 3: Group pages by patient
    unique_patient_texts = group_pages_by_patient(pages_text)

    print(f"\n🎯 Total Unique Patients Identified: {len(unique_patient_texts)}")

    return unique_patient_texts

import base64

pdf_path = "/media/bharath/DATA_8TB1/Bharath/Problem_Statement_2_630a8c8cb6/2_Submission_Input_Data_7e33bbd2e6/2_Submission Input Data/Diagnostic Report/Test 2.pdf"

with open(pdf_path, "rb") as pdf_file:
    pdf_bytes = pdf_file.read()
    pdf_base64 = base64.b64encode(pdf_bytes).decode("utf-8")

unique_patient_lists = process_pdf_and_group_patients(pdf_path)

In [ ]:
import json
import torch
import re
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

def extract_json_array(text: str):
    if not text or not text.strip():
        return []
    
    # Remove markdown code blocks
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    text = text.strip()
    
    # Try to parse as JSON array first
    try:
        result = json.loads(text)
        if isinstance(result, list):
            return result
        elif isinstance(result, dict):
            return [result]
    except json.JSONDecodeError:
        pass
    
    # Enhanced fallback: extract complete JSON array from incomplete/truncated text
    # Look for content between first [ and last ]
    start_bracket = text.find('[')
    end_bracket = text.rfind(']')
    
    if start_bracket != -1 and end_bracket != -1 and end_bracket > start_bracket:
        json_str = text[start_bracket:end_bracket+1]
        try:
            result = json.loads(json_str)
            if isinstance(result, list):
                return result
            elif isinstance(result, dict):
                return [result]
        except json.JSONDecodeError:
            pass
    
    # Streaming decoder approach - handles truncated JSON better
    decoder = json.JSONDecoder()
    observations = []
    idx = 0
    
    while idx < len(text):
        try:
            obj, end = decoder.raw_decode(text[idx:])
            if isinstance(obj, dict) and obj.get('resource', {}).get('resourceType') == 'Observation':
                observations.append(obj)
            idx += end
        except json.JSONDecodeError as e:
            # Skip malformed content and continue
            idx += 1
    
    return observations if observations else []

llm = ChatOllama(
      model="qwen2.5:32b",
      temperature=0,
      num_predict=5000,   # equivalent to max_new_tokens
      num_ctx=32768
  )

for i, raw_text in enumerate(unique_patient_lists):
    
    # 2. Define the refined Prompt with UCUM and Numerical Range Rules
    system_instructions = """### TASK: Convert Clinical Lab Report into HL7 FHIR Observation Entries (Strict Mode)

    You must extract ALL laboratory tests from the input text and convert EACH test into a separate FHIR Observation entry.

    ⚠️ CRITICAL: 
    Return ONLY a valid JSON array.
    Do NOT include markdown, explanations, commentary, or extra text.
    Output must start with "[" and end with "]".

    -------------------------------------------------------
    FHIR STRUCTURE REQUIREMENTS (MANDATORY)
    -------------------------------------------------------

    Each test must be formatted EXACTLY as:

    {
      "fullUrl": "urn:uuid:<generated-uuid>",
      "resource": {
        "resourceType": "Observation",
        "id": "<same-generated-uuid>",
        "status": "final",
        "code": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "<accurate-loinc-code>",
              "display": "<official LOINC display name>"
            }
          ],
          "text": "<EXACT test name as written in report>"
        },
        "valueQuantity": {
          "value": <numeric-value>,
          "unit": "<UCUM unit>",
          "system": "http://unitsofmeasure.org",
          "code": "<UCUM unit code>"
        },
        "referenceRange": [
          {
            "low": {
              "value": <numeric-low>,
              "unit": "<UCUM unit>",
              "system": "http://unitsofmeasure.org",
              "code": "<UCUM unit code>"
            },
            "high": {
              "value": <numeric-high>,
              "unit": "<UCUM unit>",
              "system": "http://unitsofmeasure.org",
              "code": "<UCUM unit code>"
            }
          }
        ]
      }
    }

    -------------------------------------------------------
    STRICT RULES
    -------------------------------------------------------

    1. Output MUST be a JSON array of Observation entries.
    2. Generate a unique RFC-4122 UUID for every test.
    3. The UUID in "fullUrl" MUST match the "id".
    4. Use correct and accurate LOINC codes corresponding to the test.
    5. The LOINC display must be the official display string for that code.
    6. Do NOT guess LOINC codes. Only use correct mappings.
    7. If no correct LOINC code exists, omit coding and only use "text".
    8. Use EXACT test names in "code.text" (verbatim from report).
    9. Extract EVERY test including Absolute counts.
    10. Preserve numeric precision exactly as written.
    11. Convert reference ranges like "01 - 06" → 1.0 and 6.0.
    12. UCUM NORMALIZATION:
        - "g/dl" → "g/dL"
        - "x10^6 /μL" → "10*6/uL"
        - "x10^3 /μL" → "10*3/uL"
    13. Units MUST include:
        "system": "http://unitsofmeasure.org"
    14. LOINC MUST use:
        "system": "http://loinc.org"
    15. Do NOT modify system URLs.
    16. Do NOT include null fields.
    17. Do NOT skip any test.
    18. If reference range missing, omit "referenceRange".
    19. If unit missing, omit valueQuantity.unit but keep value.
    20. Do NOT wrap output in markdown.

    -------------------------------------------------------
    FINAL OUTPUT:
    Return ONLY the JSON array of Observation entries.
    """

    full_prompt = f"{system_instructions}\n\n### Input Text :\n{raw_text}\n\n### JSON Output:"

  
    # 4. Generate Output using Ollama
    response = llm.invoke([HumanMessage(content=full_prompt)])
    raw_output = response.content.strip()
  
    final_json = extract_json_array(raw_output)

    with open(f'extracted_data_{i}_test_2.json', 'w') as f:
        json.dump(final_json, f, indent=2)

    print(f"Extraction complete. Data saved to extracted_data_{i}.json")


In [ ]:
import json
import uuid
import copy

test_folder = "/media/bharath/DATA_8TB1/Bharath/Problem_Statement_3_e4cc9a3eb9/Test_1-20260301T091620Z-1-001/Test_1/"

for i in range(5):

    # --- 1. Load Extracted Observations ---
    observation_file = f'extracted_data_{i}.json'
    with open(observation_file, 'r') as f:
        new_observations_list = json.load(f)

    # --- 2. Load Base Bundle ---
    with open(
        f'/media/bharath/DATA_8TB1/Bharath/Problem_Statement_3_e4cc9a3eb9/Test_1-20260301T091620Z-1-001/Test_1/FHIR_BUNDLE_DiagnosticReportRecord_Patient{i+1}.json',
        'r'
    ) as f:
        bundle = json.load(f)

    # --- 3. Find Composition + DiagnosticReport ---
    composition = None
    diagnostic_report = None

    for entry in bundle['entry']:
        resource_type = entry['resource']['resourceType']
        if resource_type == 'Composition':
            composition = entry['resource']
        elif resource_type in ['DiagnosticReport', 'DiagnosticReportLab']:
            diagnostic_report = entry['resource']

    # --- 4. Collect Existing Observations ---
    existing_observations = [
        entry for entry in bundle['entry']
        if entry['resource']['resourceType'] == 'Observation'
    ]

    # --- 5. Duplicate Detection (YOUR ORIGINAL LOGIC) ---
    seen_signatures = set()
    unique_observations = []

    def get_observation_signature(obs_entry):
        code_text = obs_entry['resource']['code'].get('text', '').strip().lower()
        if 'valueQuantity' in obs_entry['resource'] and \
           'value' in obs_entry['resource']['valueQuantity']:
            value = obs_entry['resource']['valueQuantity']['value']
            return f"{code_text}|{value}"
        return code_text

    # Process existing first
    for obs_entry in existing_observations:
        signature = get_observation_signature(obs_entry)
        if signature not in seen_signatures:
            seen_signatures.add(signature)
            unique_observations.append(obs_entry)

    # Process new
    for obs_entry in new_observations_list:
        signature = get_observation_signature(obs_entry)
        if signature not in seen_signatures:
            seen_signatures.add(signature)

            # Important: deep copy before modifying
            obs_copy = copy.deepcopy(obs_entry)

            # Generate NEW UUID only for new ones
            new_id = str(uuid.uuid4())
            obs_copy['resource']['id'] = new_id
            obs_copy['fullUrl'] = f"urn:uuid:{new_id}"

            unique_observations.append(obs_copy)

    # --- 6. Remove ALL Old Observations ---
    bundle['entry'] = [
        entry for entry in bundle['entry']
        if entry['resource']['resourceType'] != 'Observation'
    ]

    # --- 7. Append ALL Unique Observations ---
    bundle['entry'].extend(unique_observations)

    # --- 8. Build Proper References ---
    observation_references = []

    for obs_entry in unique_observations:
        obs_id = obs_entry['resource']['id']
        observation_references.append({
            "reference": f"urn:uuid:{obs_id}"
        })

    # --- 9. Update Composition ---
    if composition:
        if 'section' in composition and len(composition['section']) > 0:
            # Usually CBC Results section is index 2
            composition['section'][-1]['entry'] = observation_references
        else:
            composition['section'] = [{
                "title": "Lab Results",
                "entry": observation_references
            }]

    # --- 10. Update DiagnosticReport ---
    if diagnostic_report:
        diagnostic_report['result'] = observation_references

    # --- 11. Save Final Bundle ---
    output_file = f'our_final_production_bundle_patient_{i+1}.json'
    with open(output_file, 'w') as f:
        json.dump(bundle, f, indent=2)

    print(
        f"Patient {i+1} → "
        f"{len(existing_observations)} existing + "
        f"{len(new_observations_list)} new → "
        f"{len(unique_observations)} unique. "
        f"Saved: {output_file}"
    )

In [ ]:
import json
import os

# Folder where your final bundles are stored
bundle_folder = "."

for i in range(5):  # Adjust if needed

    file_name = f"our_final_production_bundle_patient_{i+1}.json"

    if not os.path.exists(file_name):
        print(f"File not found: {file_name}")
        continue

    with open(file_name, "r") as f:
        bundle = json.load(f)

    if "entry" not in bundle:
        print(f"No entries found in {file_name}")
        continue

    document_reference_entry = None
    document_reference_index = None

    # --- 1. Locate DocumentReference ---
    for idx, entry in enumerate(bundle["entry"]):
        resource = entry.get("resource", {})
        if resource.get("resourceType") == "DocumentReference":
            document_reference_entry = entry
            document_reference_index = idx
            break

    if document_reference_entry is None:
        print(f"No DocumentReference found in {file_name}")
        continue

    # --- 2. Remove from current position ---
    del bundle["entry"][document_reference_index]

    # --- 3. Append as last entry ---
    bundle["entry"].append(document_reference_entry)

    # --- 4. Save updated bundle ---
    with open(file_name, "w") as f:
        json.dump(bundle, f, indent=2)

    print(f"{file_name} → DocumentReference moved to last position successfully.")

In [ ]:
import json
import csv
import os


def flatten_json(data, parent_key=""):
    """
    Recursively flatten nested JSON into key-path mappings.
    """
    items = []

    if isinstance(data, dict):
        for k, v in data.items():
            new_key = f"{parent_key}.{k}" if parent_key else k
            items.extend(flatten_json(v, new_key))

    elif isinstance(data, list):
        for i, v in enumerate(data):
            new_key = f"{parent_key}[{i}]"
            items.extend(flatten_json(v, new_key))

    else:
        items.append((parent_key, data))

    return items


def process_resource(resource, rows):
    """
    Process a single resource and append flattened mappings.
    If the resource is a nested Bundle, process recursively.
    """
    resource_type = resource.get("resourceType", "")
    resource_id = resource.get("id", "")

    # If nested Bundle → process entries recursively
    if resource_type == "Bundle" and "entry" in resource:
        for entry in resource.get("entry", []):
            nested_resource = entry.get("resource", {})
            process_resource(nested_resource, rows)
    else:
        flattened = flatten_json(resource)
        for key, value in flattened:
            rows.append([
                resource_type,
                resource_id,
                key,
                value
            ])


def convert_bundle_json_to_csv(json_file_path, csv_file_path):
    # Load JSON
    with open(json_file_path, "r", encoding="utf-8") as f:
        bundle = json.load(f)

    if bundle.get("resourceType") != "Bundle":
        print("Not a FHIR Bundle.")
        return

    rows = []

    # Process top-level entries
    for entry in bundle.get("entry", []):
        resource = entry.get("resource", {})
        process_resource(resource, rows)

    # Ensure output directory exists
    os.makedirs(os.path.dirname(csv_file_path), exist_ok=True)

    # Write CSV
    with open(csv_file_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["ResourceType", "ResourceID", "JSON Path", "Value"])
        writer.writerows(rows)

    print(f"Successfully converted JSON to CSV")


# ===== Execution for Uploaded InsurancePlanBundle =====

input_json = "/media/bharath/DATA_8TB1/Bharath/Problem_Statement_3_e4cc9a3eb9/our_final_production_bundle_patient_5.json"
output_csv = "/media/bharath/DATA_8TB1/Bharath/Problem_Statement_3_e4cc9a3eb9/FHIR_BUNDLE_DiagnosticReportRecord_Patient5.csv"

convert_bundle_json_to_csv(input_json, output_csv)

# DONE

## Below are the future continuition codes

In [ ]:

import json
def clean_and_reorder_bundle(input_file, output_file):
    # 1. Load the JSON
    with open(input_file, 'r') as f:
        bundle = json.load(f)

    entries = bundle.get("entry", [])
    
    # Identify indices for removal and relocation
    composition_entry = None
    cleaned_entries = []

    for entry in entries:
        resource = entry.get("resource", {})
        res_type = resource.get("resourceType")

        # Task A: Find the Composition to move it later
        if res_type == "Composition":
            composition_entry = entry
        
        # Task B: Identify and skip the fake 'DocumentBundle' resource
        elif res_type == "DocumentBundle":
            print(f"🗑️ Removing invalid 'DocumentBundle' resource (ID: {resource.get('id')})")
            continue
            
        else:
            cleaned_entries.append(entry)

    # Task C: Reassemble with Composition at the very beginning
    if composition_entry:
        final_entries = [composition_entry] + cleaned_entries
        bundle["entry"] = final_entries
        # print("✅ Success: Composition moved to index 0.")
    else:
        bundle["entry"] = cleaned_entries
        # print("⚠️ Warning: No Composition resource was found to relocate.")

    # 3. Save the corrected JSON
    with open(output_file, 'w') as f:
        json.dump(bundle, f, indent=2)

clean_and_reorder_bundle("nhcx_insuranceplanbundle_bundle.json", "nhcx_insuranceplanbundle_bundle_cleaned.json")



In [ ]:
import base64

pdf_path = "nhcx_insuranceplanbundle_bundle_cleaned.json"

with open(pdf_path, "rb") as pdf_file:
    pdf_bytes = pdf_file.read()
    pdf_base64 = base64.b64encode(pdf_bytes).decode("utf-8")

def document_reference_node(input_file, output_file, pdf_base64):

    with open(input_file, 'r') as f:
        bundle = json.load(f)

    updated = False

    for entry in bundle.get("entry", []):
        resource = entry.get("resource", {})

        if resource.get("resourceType") == "DocumentReference":

            # If content does NOT exist → create it
            if "content" not in resource or not resource["content"]:

                resource["content"] = [
                    {
                        "attachment": {
                            "contentType": "application/pdf",
                            "data": pdf_base64
                        }
                    }
                ]
                print(resource["content"][0]['attachment']['data'])
                # print("Created new content block")

            else:
                # Content exists → update attachment
                attachment = resource["content"][0].setdefault("attachment", {})

                attachment["contentType"] = "application/pdf"
                attachment["data"] = pdf_base64
                print(attachment['data'])
                # print("Updated existing content block")

    with open(output_file, 'w') as f:
        json.dump(bundle, f, indent=2)

document_reference_node("nhcx_insuranceplanbundle_bundle_cleaned.json", "nhcx_insuranceplanbundle_bundle_cleaned_doctreferReady.json", pdf_base64=pdf_base64)


In [ ]:
import json

def clean_and_reorder_bundle(input_file, output_file):
    # 1. Load the JSON
    with open(input_file, 'r') as f:
        bundle = json.load(f)

    entries = bundle.get("entry", [])
    
    # Identify indices for removal and relocation
    composition_entry = None
    cleaned_entries = []

    for entry in entries:
        resource = entry.get("resource", {})
        res_type = resource.get("resourceType")

        # Task A: Find the Composition to move it later
        if res_type == "Composition":
            composition_entry = entry
        
        # Task B: Identify and skip the fake 'DocumentBundle' resource
        elif res_type == "DocumentBundle":
            print(f"🗑️ Removing invalid 'DocumentBundle' resource (ID: {resource.get('id')})")
            continue
            
        else:
            cleaned_entries.append(entry)

    # Task C: Reassemble with Composition at the very beginning
    if composition_entry:
        final_entries = [composition_entry] + cleaned_entries
        bundle["entry"] = final_entries
        print("✅ Success: Composition moved to index 0.")
    else:
        bundle["entry"] = cleaned_entries
        print("⚠️ Warning: No Composition resource was found to relocate.")

    # 3. Save the corrected JSON
    with open(output_file, 'w') as f:
        json.dump(bundle, f, indent=2)
    
    print(f"💾 Corrected file saved as: {output_file}")

# Execution
clean_and_reorder_bundle('dynamic_abdm_bundle_discharge_summary.json', 'fixed_abdm_bundle_discharge_summary.json')

In [ ]:
import json
with open('fixed_abdm_bundle_discharge_summary.json', 'r') as f:
    bundle_json = json.load(f)
original_text = extracted_text

In [ ]:
import json
import re

# --- REQUIRED PLACEHOLDERS (MUST EXIST AT RUNTIME) ---
# llm = ...
# original_text = ...
# bundle_json = ...

# --- ABDM CORRECTION PROMPT (hardcoded) ---
# correction_prompt = f"""
# SYSTEM: You are ABDM FHIR VALIDATION EXPERT. Return ONLY the corrected complete ABDM Bundle JSON.

# MANDATORY FIXES (in order):
# 1. Composition FIRST (entry[0], resourceType="Composition", correct NDHM profile)
# 2. DocumentReference LAST (all DocumentReference at end)  
# 3. ALL REFERENCES RESOLVE (urn:uuid: refs exist in bundle.entry)
# 4. TERMINOLOGY: LOINC(http://loinc.org), SNOMED(http://snomed.info/sct), UCUM(http://unitsofmeasure.org)
# 5. PROFILES: https://nrces.in/ndhm/fhir/r4/StructureDefinition/*
# 6. CONTENT: All clinical values from original_text present
# 7. MANDATORY: status, subject, issued, effectiveDateTime
# 8. NO DATA LOSS - only fix/add, never delete clinical content

# INPUT:
# ORIGINAL TEXT (SOURCE):
# {original_text}

# BUNDLE TO CORRECT:
# {json.dumps(bundle_json, indent=2)}

# RULES:
# - Return ONLY valid JSON Bundle (no markdown, no explanations)
# - Print "FIXED: description" for each change made
# - First entry MUST be Composition
# - Last entries MUST be DocumentReference  
# - All references must resolve within bundle
# - Preserve ALL clinical data from original_text

# OUTPUT ONLY THE CORRECTED JSON BUNDLE:
# """


correction_prompt = f"""
SYSTEM: You are ABDM FHIR VALIDATION EXPERT. Return ONLY the corrected complete ABDM Bundle JSON.


INPUT:
ORIGINAL TEXT (SOURCE):
{original_text}

BUNDLE TO CORRECT:
{json.dumps(bundle_json, indent=2)}


MANDATORY FIXES (in order):
1. Composition FIRST (entry[0], resourceType="Composition", correct NDHM profile)
2. DocumentReference LAST (all DocumentReference at end)  
3. ALL REFERENCES RESOLVE (urn:uuid: refs exist in bundle.entry)
4. TERMINOLOGY: LOINC(http://loinc.org), SNOMED(http://snomed.info/sct), UCUM(http://unitsofmeasure.org)
5. PROFILES: https://nrces.in/ndhm/fhir/r4/StructureDefinition/*
6. CONTENT: All clinical values from original_text present
7. MANDATORY: status, subject, issued, effectiveDateTime
8. NO DATA LOSS - only fix/add, never delete clinical content

STRICT DATA PERSISTENCE RULES:
- ZERO DELETION POLICY: You are strictly forbidden from removing any Observation, Condition, or Clinical resource present in the "BUNDLE TO CORRECT". 
- SURGICAL REPAIR ONLY: If a resource is invalid, fix its structure, add missing mandatory fields (like 'system' or 'code'), and align it with the profile. DO NOT discard the resource.
- AUGMENTATION: If the "ORIGINAL TEXT" contains clinical information (e.g., blood pressure, symptoms, dates) missing from the "BUNDLE TO CORRECT", you MUST add them as new Observations or fill in the empty fields.
- OBSERVATION INTEGRITY: Ensure every 'valueQuantity' or 'valueCodeableConcept' from the original input is preserved. If the validator says a field is "extra" or "not allowed", move that data into a 'note' or 'comment' field rather than deleting it.


RULES:
- Return ONLY valid JSON Bundle (no markdown, no explanations)
- First entry MUST be Composition
- Last entries MUST be DocumentReference  
- All references must resolve within bundle
- Preserve ALL clinical data from original_text and ALL existing resources from the input JSON.

OUTPUT ONLY THE CORRECTED JSON BUNDLE:
"""


# ✅ RUN CORRECTION
print("🔍 Starting ABDM Bundle Correction...")

result = llm.invoke(correction_prompt)

print("\n" + "=" * 80)
print("🎯 CORRECTION PROCESS COMPLETE")
print("=" * 80)


In [ ]:
def extract_json(text: str):
    if not text or not text.strip():
        return None
    
    # Remove markdown code blocks
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    text = text.strip()
    
    decoder = json.JSONDecoder()
    idx = 0
    
    while idx < len(text):
        try:
            obj, end = decoder.raw_decode(text[idx:])
            if isinstance(obj, str):
                try:
                    obj = json.loads(obj)
                except:
                    pass
            return obj
        except json.JSONDecodeError:
            idx += 1
    return None


# --- EXTRACT AND SAVE FINAL BUNDLE ---
try:
    # Clean any markdown code blocks
    json_obj = extract_json(result.content)

    if json_obj is None:
        raise ValueError("No valid JSON found in model output")

    final_bundle = json_obj
    
    # Save validated bundle
    with open("validated_abdm_bundle_discharge_summary.json", "w") as f:
        json.dump(final_bundle, f, indent=2)
    
    print("Json is Extracted well....")
    entries = final_bundle.get('entry', [])
    comp_type = entries['resource']['resourceType'] if entries else "Empty"
    last_type = entries[-1]['resource']['resourceType'] if entries else "Empty"
    
    print(f"✅ SAVED: validated_abdm_bundle_discharge_summary.json")
    print(f"📊 Total Entries: {len(entries)}")
    print(f"🎯 First Resource: {comp_type} (Should be Composition)")
    print(f"📄 Last Resource: {last_type} (Should be DocumentReference)")
    
    print("\n📋 Bundle Structure OK ✅")
    
except json.JSONDecodeError as e:
    print("❌ JSON parsing failed")
    print(f"Error: {e}")
    print("\nRaw LLM Output:")
    print(result.content)
    
except Exception as e:
    print(f"❌ Unexpected error: {e}")
    print("\nRaw LLM Output:")
    print(result.content)


In [ ]:
import subprocess

cmd = [
    "java",
    "-jar",
    "validator_cli.jar",
    "healed_abdm_bundle_discharge_summary.json",
    "-version",
    "4.0.1"
]

try:
    # Adding check=True will raise an exception if the validator fails
    result = subprocess.run(cmd, capture_output=True, text=True, check=False)

    print("--- VALIDATION OUTPUT ---")
    print(result.stdout)

    if result.stderr:
        print("--- SYSTEM/JAVA ERRORS ---")
        print(result.stderr)

except FileNotFoundError:
    print("Error: 'java' is not installed or 'validator_cli.jar' is missing from this directory.")

In [ ]:
import re

raw_output = result.stdout  # your full validator console text

# Remove ANSI escape sequences
clean = re.sub(r'\x1B\[[0-?]*[ -/]*[@-~]', '', raw_output)

# Extract only error lines
errors = []
for line in clean.splitlines():
    line = line.strip()
    if line.startswith("Error @"):
        errors.append(line)

validation_report = "\n".join(errors)

print(validation_report)

In [ ]:
import subprocess
import json
import os

# --- 1. RUN THE HL7 VALIDATOR ---
print("🔬 Step 1: Running HL7 FHIR Validator...")

# We use the IG for ABDM to ensure the validator knows the Indian profiles
# Replace 'nrces.in.ndhm#6.0.0' with your specific version if different
to_validate_path = "validated_abdm_bundle_discharge_summary.json"
healed_path = "healed_abdm_bundle_discharge_summary.json"

cmd = [
    "java", "-Xmx2G", "-jar", "validator_cli.jar",
    to_validate_path, 
    "-version", "4.0.1",
    "-ig", "nrces.in.ndhm#6.0.0" 
]

validator_result = subprocess.run(cmd, capture_output=True, text=True)
validation_report = validator_result.stdout


import re

raw_output = validation_report  # your full validator console text

# Remove ANSI escape sequences
clean = re.sub(r'\x1B\[[0-?]*[ -/]*[@-~]', '', raw_output)

# Extract only error lines
errors = []
for line in clean.splitlines():
    line = line.strip()
    if line.startswith("Error @"):
        errors.append(line)

validation_report = "\n".join(errors)

# print(validation_report)


print("--- VALIDATOR FEEDBACK CAPTURED ---")

with open(to_validate_path, 'r') as f:
    bundle_json = json.load(f)

# --- 2. PREPARE THE POWERFUL REPAIR PROMPT ---
# We combine the Original Text, the Flawed JSON, and the Error Report
repair_prompt = f"""
Extract ONLY a valid HL7 FHIR R4 Bundle from the provided failed bundle by fixing validation errors.

SOURCE CLINICAL TEXT (for reference only):
{original_text}

FAILED BUNDLE (currently invalid):
{json.dumps(bundle_json, indent=2)}

VALIDATOR ERROR LOG (MUST FIX THESE):
{validation_report}

CRITICAL REPAIR RULES (NON-NEGOTIABLE):

• FIX ONLY errors flagged in VALIDATOR ERROR LOG
• NEVER DELETE any existing clinical data, Observations, components, or extensions
• If field has data → KEEP IT, just fix format/profile/reference
• If validator flags missing required field → ADD with minimal valid value from source text
• PRESERVE all existing 'text', 'display', 'value', and clinical measurements exactly
• Fix terminology by using validator suggestions or source text mappings (keep display intact)

MANDATORY ABDM BUNDLE REQUIREMENTS:
• First entry MUST be Composition with status, subject, type
• All references "urn:uuid:..." MUST resolve internally
• Every resource MUST have valid "id" (UUID format)
• meta.profile MUST match ABDM StructureDefinitions

SPECIFIC FIXES BY ERROR TYPE:
PATHOMAP ERRORS → Fix element cardinality/conformance using source text
TERMINOLOGY ERRORS → Map to valid SNOMED/LOINC from source text (preserve display)
PROFILE ERRORS → Add correct meta.profile from ABDM specs
MISSING REQUIRED → Add minimal valid value (status: "final", subject reference, etc.)
INVALID UUID → Generate proper RFC-4122 UUID but preserve reference consistency

PRESERVATION RULES:
• DO NOT remove any existing Observation components or values
• DO NOT modify numeric precision or dates
• DO NOT change clinical meaning or measurements
• Unknown fields → LEAVE UNTOUCHED

OUTPUT FORMAT:
Return ONLY the corrected valid JSON Bundle starting with {{

"""

# --- 3. INVOKE LLM FOR CORRECTION ---
print("🤖 Step 2: LLM is repairing the Bundle based on Validator Errors...")
correction_response = llm.invoke(repair_prompt)

# --- 4. PARSE AND SAVE THE HEALED BUNDLE ---
try:
    # Extract JSON from potential Markdown formatting
    raw_content = correction_response.content.strip()
    if "```json" in raw_content:
        clean_json = raw_content.split("```json")[1].split("```")[0].strip()
    elif "```" in raw_content:
        clean_json = raw_content.split("```")[1].split("```")[0].strip()
    else:
        clean_json = raw_content

    healed_bundle = json.loads(clean_json)
    
    with open(healed_path, "w") as f:
        json.dump(healed_bundle, f, indent=2)
        
    print(f"\n✅ SUCCESS: Healed bundle saved to {healed_path}")
    
    # Quick Structure Check
    first_res = healed_bundle['entry'][0]['resource']['resourceType']
    print(f"📊 New Bundle Entry Count: {len(healed_bundle.get('entry', []))}")
    print(f"🎯 First Resource check: {first_res}")

except Exception as e:
    print(f"❌ Failed to parse healed JSON: {e}")
    print("Full LLM Output for debugging:")
    print(correction_response.content)

In [ ]:
import subprocess
import json
import os
from typing import TypedDict, Annotated, Sequence
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import operator

# State definition for the workflow
class AgentState(TypedDict):
    bundle_json: dict
    validation_report: str
    original_text: str
    iteration: int
    max_iterations: int
    healed_path: str
    success: bool

# Initialize state
def init_state():
    to_validate_path = "validated_abdm_bundle_discharge_summary.json"
    healed_path = "healed_abdm_bundle_discharge_summary_3agents.json"
    
    with open(to_validate_path, 'r') as f:
        initial_bundle = json.load(f)
    
    # with open("original_clinical_text.txt", 'r') as f:  # Assume you have this file
    #     original_text = f.read()
    
    return {
        "bundle_json": initial_bundle,
        "validation_report": "",
        "original_text": original_text,
        "iteration": 0,
        "max_iterations": 3,
        "healed_path": healed_path,
        "success": False
    }

# Step 1: HL7 FHIR Validator Agent
def validator_agent(state: AgentState) -> AgentState:
    print(f"🔬 Iteration {state['iteration'] + 1}: Running HL7 FHIR Validator...")
    
    # Save current bundle for validation
    temp_validate_path = f"temp_validate_iter_{state['iteration']}.json"
    with open(temp_validate_path, "w") as f:
        json.dump(state['bundle_json'], f, indent=2)
    
    # Run validator
    cmd = [
        "java", "-Xmx2G", "-jar", "validator_cli.jar",
        temp_validate_path,
        "-version", "4.0.1",
        "-ig", "nrces.in.ndhm#6.0.0"
    ]
    
    validator_result = subprocess.run(cmd, capture_output=True, text=True)
    validation_report = validator_result.stdout

    import re

    raw_output = validation_report  # your full validator console text

    # Remove ANSI escape sequences
    clean = re.sub(r'\x1B\[[0-?]*[ -/]*[@-~]', '', raw_output)

    # Extract only error lines
    errors = []
    for line in clean.splitlines():
        line = line.strip()
        if line.startswith("Error @"):
            errors.append(line)

    validation_report = "\n".join(errors)
    
    print("✅ Validator feedback captured")
    
    # Update state
    state['validation_report'] = validation_report
    state['iteration'] += 1
    
    return state

# Step 2: LLM Surgical Repair Agent (same prompt for all iterations)
repair_prompt_template = """
Extract ONLY a valid HL7 FHIR R4 Bundle from the provided failed bundle by fixing validation errors.

SOURCE CLINICAL TEXT (for reference only):
{original_text}

FAILED BUNDLE (currently invalid):
{failed_bundle}

VALIDATOR ERROR LOG (MUST FIX THESE):
{validation_report}

CRITICAL REPAIR RULES:
• FIX ONLY errors flagged in VALIDATOR ERROR LOG  
• NEVER DELETE any existing clinical data, Observations, components, or extensions
• PRESERVE all existing 'text', 'display', 'value', and clinical measurements exactly

OUTPUT FORMAT:
Return ONLY the corrected valid JSON Bundle starting with {{
"""

def repair_agent(state: AgentState) -> AgentState:
    print(f"🤖 Iteration {state['iteration']}: LLM repairing Bundle...")
    
    prompt = repair_prompt_template.format(
        original_text=state['original_text'],
        failed_bundle=json.dumps(state['bundle_json'], indent=2),
        validation_report=state['validation_report']
    )
    
    messages = [HumanMessage(content=prompt)]
    correction_response = llm.invoke(messages)
    
    # FIXED JSON EXTRACTION
    raw_content = correction_response.content.strip()
    print(f"🔍 LLM Raw Output Preview: {raw_content[:200]}...")

    # Use existing extract_json function
    healed_bundle = extract_json(raw_content)

    if healed_bundle:
        print(f"✅ Parsed JSON keys: {list(healed_bundle.keys())[:5]}...")
    else:
        print("❌ Failed to extract JSON")
        return state

    
    try:
        # healed_bundle = json.loads(clean_json)
        print(f"✅ Parsed JSON keys: {list(healed_bundle.keys())[:5]}...")
        
        # CRITICAL: Check if bundle actually changed
        old_hash = hash(json.dumps(state['bundle_json'], sort_keys=True))
        new_hash = hash(json.dumps(healed_bundle, sort_keys=True))
        if old_hash == new_hash:
            print("⚠️ WARNING: Bundle unchanged! LLM didn't fix anything.")
        else:
            print("✅ Bundle successfully modified!")
        
        state['bundle_json'] = healed_bundle
        
    except json.JSONDecodeError as e:
        print(f"❌ JSON Parse Error: {e}")
        print(f"Failed JSON content: {clean_json[:1000]}")
        # Don't overwrite on parse failure
        pass
    
    return state


# Conditional edge: Check if we should continue
def should_continue(state: AgentState) -> str:
    if state['iteration'] >= state['max_iterations']:
        return "final_save"
    
    # Quick check: If first resource is Composition and no critical errors, we can stop early
    try:
        first_entry = state['bundle_json']['entry'][0]['resource']
        if first_entry.get('resourceType') == 'Composition':
            print(f"🎯 Early stop: First resource is Composition at iteration {state['iteration']}")
            return "final_save"
    except:
        pass
    
    return "repair"

# Final save and validation
def final_save_agent(state: AgentState) -> AgentState:
    print(f"🏁 Final Save: Iteration {state['iteration']}")
    
    # Save final healed bundle
    with open(state['healed_path'], "w") as f:
        json.dump(state['bundle_json'], f, indent=2)
    
    # Final structure check
    try:
        entry_count = len(state['bundle_json'].get('entry', []))
        first_res = state['bundle_json']['entry'][0]['resource']['resourceType']
        print(f"📊 Final Bundle Entry Count: {entry_count}")
        print(f"🎯 Final First Resource: {first_res}")
        state['success'] = True
    except Exception as e:
        print(f"⚠️ Final structure check failed: {e}")
        state['success'] = False
    
    return state

# Build the LangGraph workflow
def create_workflow():
    workflow = StateGraph(AgentState)
    
    # Add nodes
    workflow.add_node("validate", validator_agent)
    workflow.add_node("repair", repair_agent)
    workflow.add_node("save", final_save_agent)
    
    # Set entry point
    workflow.set_entry_point("validate")
    
    # Add edges
    workflow.add_edge("validate", "repair")
    workflow.add_conditional_edges(
        "repair",
        should_continue,
        {
            "repair": "validate",
            "final_save": "save"
        }
    )
    workflow.add_edge("save", END)
    
    return workflow.compile()

# 🚀 RUN THE 3-AGENT WORKFLOW
print("🔬🔬🔬 ABDM FHIR 3-Agent Surgical Repair Pipeline Starting...")
print("=" * 60)

# Initialize and run
initial_state = init_state()
app = create_workflow()

final_state = app.invoke(initial_state)

print("\n" + "=" * 60)
print("🎉 PIPELINE COMPLETE!")
print(f"✅ Success: {final_state['success']}")
print(f"📁 Final file: {final_state['healed_path']}")
print(f"🔄 Total iterations: {final_state['iteration']}")


In [ ]:
validation_report